In [1]:
import sys
assert "torch_bedrock" in sys.executable, (
    "Wrong kernel. Switch to torch_bedrock via Kernel → Change Kernel."
)
print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")

Python: 3.11.11 | packaged by conda-forge | (main, Dec  5 2024, 08:47:03) [Clang 18.1.8 ]
Executable: /opt/anaconda3/envs/torch_bedrock/bin/python


In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main


In [3]:
import random
import numpy as np
import torch

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device: {DEVICE}")
print(f"[Config] Random seed: {RANDOM_SEED}")

[Config] Device: cpu
[Config] Random seed: 42


In [4]:
from Transformer.models.longformer_sequence import LongformerSequence

SEQUENCE_MODEL_CLS  = LongformerSequence
SEQUENCE_MODEL_NAME = "Longformer"
print(f"[Config] Sequence model: {SEQUENCE_MODEL_NAME}")

[Config] Sequence model: Longformer


In [5]:
# MAMBA ALTERNATIVE — uncomment this cell to train Mamba instead of Longformer.
# Comment out the Longformer cell above before running this one.
# No other changes are needed — the pipeline is fully modular.
#
# from Transformer.models.mamba_sequence import Mamba3DSequence
# SEQUENCE_MODEL_CLS  = Mamba3DSequence
# SEQUENCE_MODEL_NAME = "Mamba3D"
# print(f"[Config] Sequence model: {SEQUENCE_MODEL_NAME}")

In [6]:
from Transformer.config.model_config import ModelConfig

config = ModelConfig(
    dvf_dir=REPO_ROOT / "data" / "dvf_structured",
    survival_labels_dir=REPO_ROOT / "Baseline" / "outputs",
    tabular_path=REPO_ROOT / "Baseline" / "outputs" / "mci_tensor.npy",
    checkpoint_dir=REPO_ROOT / "Transformer" / "checkpoints",
    figures_dir=REPO_ROOT / "Transformer" / "figures",
)
config.validate()

print("Model dimensions")
print(f"d_model: {config.d_model}")
print(f"Tokens per visit: {config.n_tokens_per_visit} (8x8x8 BrainIAC grid)")
print(f"Max visits per subject: {config.v_max}")
print(f"Temporal grid: {config.t_grid} months")
print(f"Competing risks: {config.n_risks}  (dementia, mortality)")
print(f"PMA seed vectors: {config.pma_seeds}")
eff = config.batch_size_physical * config.gradient_accumulation_steps
print(f"Effective batch size: {eff}({config.batch_size_physical} x {config.gradient_accumulation_steps})")

Model dimensions
d_model: 512
Tokens per visit: 512 (8x8x8 BrainIAC grid)
Max visits per subject: 10
Temporal grid: [12, 24, 36, 48, 60] months
Competing risks: 2  (dementia, mortality)
PMA seed vectors: 8
Effective batch size: 16(2 x 8)


In [7]:
import os
import re

FLOW_DIR = REPO_ROOT / "data" / "adni-flow-tensors"
STRUCT_DIR = REPO_ROOT / "data" / "dvf_structured"

VISIT_MONTHS = {
    "bl": 0, "m03": 3, "m06": 6, "m12": 12, "m18": 18,
    "m24": 24, "m36": 36, "m48": 48, "m60": 60,
}

if STRUCT_DIR.exists():
    print(f"Structured dir already exists: {STRUCT_DIR}")
else:
    STRUCT_DIR.mkdir(parents=True)
    flow_files = sorted(FLOW_DIR.glob("*_flow.npy"))
    print(f"Total flow files: {len(flow_files)}")

    for f in flow_files:
        match = re.match(r'(\d+_S_\d+)_(.+)_flow', f.stem)
        if not match:
            continue
        subject_id = match.group(1)
        visit_code = match.group(2)
        month = VISIT_MONTHS.get(visit_code)
        if month is None:
            continue

        subj_dir = STRUCT_DIR / subject_id
        subj_dir.mkdir(exist_ok=True)
        dst = subj_dir / f"{month:03d}.npy"
        if not dst.exists():
            os.symlink(str(f.resolve()), str(dst))

    n_subjects = sum(1 for d in STRUCT_DIR.iterdir() if d.is_dir())
    print(f"Structured {n_subjects} subject directories in {STRUCT_DIR}")

Structured dir already exists: /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main/data/dvf_structured


In [8]:
durations_raw = np.load(str(config.survival_labels_dir / "mci_y_duration.npy"))
events_raw = np.load(str(config.survival_labels_dir / "mci_y_event.npy"))

# Durations in the baseline labels are in years — convert to months
# to match t_grid=[12,24,36,48,60] which is in months
if durations_raw.max() < 20:
    print("Durations appear to be in years — converting to months")
    durations_raw = durations_raw * 12.0

print(f"Subjects: {len(durations_raw)}")
print(f"Event rate: {(events_raw > 0).mean():.1%}")
print(f"Dementia events: {(events_raw == 1).sum()}")
print(f"Mortality events: {(events_raw == 2).sum()}")
print(f"Censored: {(events_raw == 0).sum()}")
print(f"Duration range: {durations_raw.min():.1f} – {durations_raw.max():.1f} months")

Durations appear to be in years — converting to months
Subjects: 958
Event rate: 40.2%
Dementia events: 385
Mortality events: 0
Censored: 573
Duration range: 1.0 – 199.1 months


In [9]:
from sklearn.model_selection import train_test_split

# Discover DVF subjects
dvf_subject_ids = sorted([d.name for d in config.dvf_dir.iterdir() if d.is_dir()])
print(f"DVF subjects found: {len(dvf_subject_ids)}")
print(f"Survival labels   : {len(durations_raw)}")

# We need to match — use the first N subjects that have both DVFs and labels
n_labeled = len(durations_raw)
subject_ids = dvf_subject_ids[:n_labeled]
durations = durations_raw[:n_labeled]
events = events_raw[:n_labeled]
print(f"Matched subjects  : {len(subject_ids)}")

indices = np.arange(len(subject_ids))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=(events > 0).astype(int),
)
print(f"Training subjects : {len(train_idx)}")
print(f"Validation subjs : {len(val_idx)}")
print(f"Train event rate : {(events[train_idx] > 0).mean():.1%}")
print(f"Val event rate : {(events[val_idx] > 0).mean():.1%}")

DVF subjects found: 1235
Survival labels   : 958
Matched subjects  : 958
Training subjects : 766
Validation subjs : 192
Train event rate : 40.2%
Val event rate : 40.1%


In [10]:
from Transformer.data.dvf_dataset import NormalizationStats

train_dvf_paths = []
for i in train_idx:
    sid = subject_ids[i]
    subj_dir = config.dvf_dir / sid
    train_dvf_paths.extend(sorted(subj_dir.glob("*.npy")))

print(f"Training DVF files for normalization: {len(train_dvf_paths)}")
norm_stats = NormalizationStats.compute(train_dvf_paths)

print("Per-channel normalization statistics (training split):")
for c, name in enumerate(["Δx", "Δy", "Δz"]):
    print(f"{name}: mean={norm_stats.mean[c]:.6f}  std={norm_stats.std[c]:.6f}")

Training DVF files for normalization: 2505
Per-channel normalization statistics (training split):
Δx: mean=-0.000026  std=0.500700
Δy: mean=0.007439  std=0.644005
Δz: mean=-0.009050  std=1.302763


In [11]:
from Transformer.data.dvf_dataset import LongitudinalDVFDataset
from torch.utils.data import DataLoader

# Save per-split labels so the dataset can load them
split_labels_dir = REPO_ROOT / "Transformer" / "split_labels"
split_labels_dir.mkdir(exist_ok=True)

# Training split
train_sids = [subject_ids[i] for i in train_idx]
np.save(split_labels_dir / "mci_y_duration.npy", durations[train_idx])
np.save(split_labels_dir / "mci_y_event.npy", events[train_idx])

train_dataset = LongitudinalDVFDataset(
    subject_ids=train_sids,
    dvf_dir=config.dvf_dir,
    config=config,
    norm_stats=norm_stats,
    survival_labels_dir=split_labels_dir,
)

# Validation split
val_sids = [subject_ids[i] for i in val_idx]
val_labels_dir = REPO_ROOT / "Transformer" / "val_labels"
val_labels_dir.mkdir(exist_ok=True)
np.save(val_labels_dir / "mci_y_duration.npy", durations[val_idx])
np.save(val_labels_dir / "mci_y_event.npy", events[val_idx])

val_dataset = LongitudinalDVFDataset(
    subject_ids=val_sids,
    dvf_dir=config.dvf_dir,
    config=config,
    norm_stats=norm_stats,
    survival_labels_dir=val_labels_dir,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size_physical,
    shuffle=True,
    collate_fn=LongitudinalDVFDataset.collate_fn,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size_physical,
    shuffle=False,
    collate_fn=LongitudinalDVFDataset.collate_fn,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)
print(f"Training batches : {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches : 383
Validation batches: 96


In [12]:
sample = train_dataset[0]
print(f"Sample subject: {sample['subject_id']}")
print("Data structure:")
for key, val in sample.items():
    if hasattr(val, 'shape'):
        print(f"{key:<20}: shape={list(val.shape)} dtype={val.dtype}")
    else:
        print(f"{key:<20}: {val}")

Sample subject: 057_S_1379
Data structure:
dvf_sequence        : shape=[10, 3, 128, 128, 128] dtype=torch.float32
visit_times         : shape=[10] dtype=torch.float32
time_deltas         : shape=[10] dtype=torch.float32
missing_mask        : shape=[10] dtype=torch.int64
tabular             : shape=[10, 1] dtype=torch.float32
duration            : shape=[] dtype=torch.float32
event               : shape=[] dtype=torch.int64
subject_id          : 057_S_1379


In [13]:
from Transformer.models.brainiac_extractor import BrainIACFeatureExtractor

brainiac = BrainIACFeatureExtractor(config=config, pretrained_path=None)
total = sum(p.numel() for p in brainiac.parameters())
trainable = brainiac.trainable_parameter_count()
print(f"BrainIACFeatureExtractor")
print(f"Total parameters : {total:,}")
print(f"Trainable (Stage 1) : {trainable:,}  (projection head)")
print(f"Frozen : {total - trainable:,}  (backbone)")

brainiac-model package not found — falling back to MONAI ResNet50 with layer config (3, 4, 6, 3).
No pre-trained weights provided — backbone initialised with random weights. Pre-trained BrainIAC weights are strongly recommended for convergence.


BrainIACFeatureExtractor
Total parameters : 48,828,864
Trainable (Stage 1) : 2,626,048  (projection head)
Frozen : 46,202,816  (backbone)


In [14]:
from Transformer.models.pooling import PMA
from Transformer.models.survival_head import TraCeRSurvivalHead

pma  = PMA(config)
head = TraCeRSurvivalHead(d_input=pma.output_dim(), config=config)
print(f"PMA output dim : {pma.output_dim():,}  ({config.pma_seeds} seeds x {config.d_model})")
print(f"TraCeR output : [{config.n_risks} risks x {config.n_grid} grid points]")
print(f"Temporal grid : {config.t_grid} months")

PMA output dim : 4,096  (8 seeds x 512)
TraCeR output : [2 risks x 5 grid points]
Temporal grid : [12, 24, 36, 48, 60] months


In [15]:
from Transformer.losses.ipcw_loss import CensoringSurvivalEstimator

train_durations = durations[train_idx]
train_events = events[train_idx]
val_durations = durations[val_idx]
val_events = events[val_idx]

censoring_estimator = CensoringSurvivalEstimator()
censoring_estimator.fit(train_durations, train_events)

test_times = np.array([12.0, 24.0, 36.0, 48.0, 60.0])
g_values = censoring_estimator(test_times)
print("Censoring survival G(t) at grid points:")
for t, g in zip(test_times, g_values):
    print(f"  G({t:.0f} months) = {g:.4f}")

Censoring survival G(t) at grid points:
  G(12 months) = 0.9343
  G(24 months) = 0.8137
  G(36 months) = 0.6877
  G(48 months) = 0.5286
  G(60 months) = 0.4173


In [16]:
from Transformer.utils.chi_interpolation import ConstantHazardInterpolator

chi_interpolator = ConstantHazardInterpolator.from_config(config)
print(f"CHI grid: {chi_interpolator.t_grid.tolist()} months")

CHI grid: [12.0, 24.0, 36.0, 48.0, 60.0] months


In [17]:
from Transformer.models.pipeline import ADNISurvivalPipeline

pipeline = ADNISurvivalPipeline(
    config=config,
    sequence_model_cls=SEQUENCE_MODEL_CLS,
    pretrained_brainiac_path=None,
).to(DEVICE)

total_params = sum(p.numel() for p in pipeline.parameters())
print(f"Pipeline: {SEQUENCE_MODEL_NAME} backend")
print(f"  Total parameters: {total_params:,}")

brainiac-model package not found — falling back to MONAI ResNet50 with layer config (3, 4, 6, 3).
No pre-trained weights provided — backbone initialised with random weights. Pre-trained BrainIAC weights are strongly recommended for convergence.


Pipeline: Longformer backend
  Total parameters: 94,192,202


In [18]:
from Transformer.training.trainer import LPFTTrainer

trainer = LPFTTrainer(
    model=pipeline,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    censoring_estimator=censoring_estimator,
    chi_interpolator=chi_interpolator,
    train_durations=train_durations,
    train_events=train_events,
    norm_stats=norm_stats,
    device=str(DEVICE),
)
print("LPFTTrainer ready.")
print(f"Effective batch : {config.batch_size_physical * config.gradient_accumulation_steps}")
print(f"Stage 1 epochs : {config.max_epochs_stage1}")
print(f"Stage 2 max : {config.max_epochs_stage2}")
print(f"Early stopping : patience={config.early_stopping_patience}")

LPFTTrainer ready.
Effective batch : 16
Stage 1 epochs : 5
Stage 2 max : 100
Early stopping : patience=15


In [19]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

print("STAGE 1 — Linear Probe (PMA + TraCeR head only)")
stage1_metrics = trainer.run_stage1()
print(f"\nStage 1 complete.")
print(f"Val C_td : {stage1_metrics['c_td']:.4f}")
print(f"Val IBS : {stage1_metrics['ibs']:.4f}")
print(f"Val Uno C : {stage1_metrics['uno_c']:.4f}")

2026-04-21 21:54:32,105 | INFO | ═══ Stage 1: Linear Probe (5 epochs) ═══


STAGE 1 — Linear Probe (PMA + TraCeR head only)


Input ids are automatically padded to be a multiple of `config.attention_window`: 512
2026-04-22 03:04:52,996 | INFO | Metrics — C_td=0.4226, Uno_C=0.5199 (tau=38.7), IBS=0.2068
2026-04-22 03:04:53,001 | INFO | Stage1 Epoch 1/5 — loss=nan, C_td=0.4226, Uno_C=0.5199, IBS=0.2068
2026-04-22 08:15:54,188 | INFO | Metrics — C_td=0.4178, Uno_C=0.4822 (tau=38.7), IBS=0.2203
2026-04-22 08:15:54,192 | INFO | Stage1 Epoch 2/5 — loss=2.4168, C_td=0.4178, Uno_C=0.4822, IBS=0.2203
2026-04-22 13:24:16,401 | INFO | Metrics — C_td=0.4130, Uno_C=0.4773 (tau=38.7), IBS=0.2046
2026-04-22 13:24:16,405 | INFO | Stage1 Epoch 3/5 — loss=2.4093, C_td=0.4130, Uno_C=0.4773, IBS=0.2046
2026-04-22 18:31:56,124 | INFO | Metrics — C_td=0.4108, Uno_C=0.4748 (tau=38.7), IBS=0.2368
2026-04-22 18:31:56,128 | INFO | Stage1 Epoch 4/5 — loss=2.4073, C_td=0.4108, Uno_C=0.4748, IBS=0.2368


KeyboardInterrupt: 

In [ ]:
print("STAGE 2 — Fine-Tuning (sequence + PMA + head)")
stage2_metrics = trainer.run_stage2()
print(f"Stage 2 complete.")
print(f"Best C_td: {stage2_metrics.get('c_td', 0):.4f}")
print(f"Best IBS: {stage2_metrics.get('ibs', 0):.4f}")
print(f"Best Uno C: {stage2_metrics.get('uno_c', 0):.4f}")
print(f"Best epoch: {trainer.best_epoch}")

In [ ]:
from Transformer.metrics.concordance import compute_all_metrics, format_comparison_table

# Collect validation hazards
pipeline.eval()
all_hazards = []
with torch.no_grad():
    for batch in val_loader:
        dvf_seq = batch["dvf_sequence"].to(DEVICE)
        time_deltas = batch["time_deltas"].to(DEVICE)
        missing_mask = batch["missing_mask"].to(DEVICE)
        out = pipeline(dvf_seq, time_deltas, missing_mask)
        all_hazards.append(out["hazards"].float().cpu())

val_hazards = torch.cat(all_hazards, dim=0)
print(f"Validation hazards shape: {list(val_hazards.shape)}")

final_metrics = compute_all_metrics(
    val_hazards, val_durations, val_events,
    train_durations, train_events,
    chi_interpolator, pipeline.survival_head, config,
)

print(f"Final Validation Metrics — {SEQUENCE_MODEL_NAME}")
print(f"Antolini C_td : {final_metrics['c_td']:.4f}")
print(f"Uno's C_τ: {final_metrics['uno_c']:.4f}  (τ={final_metrics['tau']:.1f}mo)")
print(f"IBS: {final_metrics['ibs']:.4f}")

In [ ]:
from IPython.display import Markdown, display

markdown_table, csv_rows = format_comparison_table(
    transformer_metrics=final_metrics,
    cohort="MCI→Dementia",
)
display(Markdown(markdown_table))

In [ ]:
import csv

output_dir = REPO_ROOT / "Transformer" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "model_comparison_transformer.csv"

with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
    writer.writeheader()
    writer.writerows(csv_rows)
print(f"Comparison table saved → {output_path}")

In [ ]:
import matplotlib.pyplot as plt

config.figures_dir.mkdir(parents=True, exist_ok=True)

sample_h = val_hazards[0:1]
with torch.no_grad():
    S = pipeline.survival_head.hazards_to_survival(sample_h)[0].numpy()
    cif = pipeline.survival_head.hazards_to_cif(sample_h)[0, 0].numpy()

t_grid = np.array(config.t_grid)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_grid, S, marker="o", color="steelblue", label="S(t)")
axes[0].set_xlabel("Months"); axes[0].set_ylabel("Probability")
axes[0].set_title("Overall Survival"); axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(t_grid, cif, marker="s", color="crimson", label="CIF Dementia")
axes[1].set_xlabel("Months"); axes[1].set_ylabel("Cumulative Incidence")
axes[1].set_title("CIF — MCI→Dementia"); axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3); axes[1].legend()

plt.tight_layout()
fig_path = config.figures_dir / "sample_survival_cif.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {fig_path}")